# C06 — Metric Learning: From Siamese Networks to FaceNet

> **Audience**: PhD students · **Framework**: PyTorch · **Dataset**: MNIST pairs

Metric learning trains a network to produce embeddings where semantically similar
inputs are close and dissimilar inputs are far apart in embedding space.

**Why not use softmax classification?**
Classification heads require a fixed set of classes seen during training.
Metric learning produces embeddings that generalise to *unseen* classes at test
time — critical for face recognition, few-shot learning, and retrieval.

**Conceptual progression**
```
Siamese Network + Contrastive Loss   → learn "similar / dissimilar" pairs
    ↓
Triplet Loss                         → harder constraint: anchor closer to positive than negative
    ↓
FaceNet / ArcFace                    → large-scale identity learning with hard mining
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print(f"Device: {DEVICE}")

# 1) Siamese Network and Contrastive Loss

**Architecture**
A Siamese network consists of two identical branches (shared weights)
that each process one input. The final loss operates on the *distance*
between the two outputs.

```
 Input A ──► Encoder ──►  z_A ─┐
                                ├─► distance d(z_A, z_B) → loss
 Input B ──► Encoder ──►  z_B ─┘
```

**Contrastive Loss** (Hadsell et al., 2006)

$$L = (1-y) \cdot \frac{d^2}{2} + y \cdot \frac{\max(0, m - d)^2}{2}$$

- $y=0$: same class pair → loss pushes $d \rightarrow 0$
- $y=1$: different class pair → loss pushes $d \geq m$ (margin)
- $m$: margin hyperparameter — how far apart dissimilar pairs must be

**Why contrastive loss can be tricky**
Easy negative pairs (clearly different classes) contribute zero loss once $d > m$.
Training saturates on these, which is why we later move to Triplet Loss
that focuses on *hard* negatives.

In [ ]:
class PairDataset(Dataset):
    """
    Generates pairs of MNIST images labelled 0 (same class) or 1 (different class).

    50% of pairs are same-class, 50% are different-class.
    """

    def __init__(self, split: str = "train", num_pairs: int = 10000) -> None:
        transform = transforms.Compose([transforms.ToTensor()])
        base = torchvision.datasets.MNIST(
            root="./data", train=(split == "train"), download=True, transform=transform
        )
        self.data   = base.data.float() / 255.0   # (N, 28, 28)
        self.labels = base.targets                 # (N,)
        self.num_pairs = num_pairs

        # Pre-build index lists per class for fast same-class sampling
        self.class_indices = {
            c: (self.labels == c).nonzero(as_tuple=True)[0].tolist()
            for c in range(10)
        }

    def __len__(self): return self.num_pairs

    def __getitem__(self, _) -> tuple:
        same_class = np.random.rand() < 0.5

        # Anchor: pick a random sample and its class
        idx_a = np.random.randint(len(self.data))
        cls_a = self.labels[idx_a].item()

        if same_class:
            idx_b = np.random.choice(self.class_indices[cls_a])
            y     = torch.tensor(0, dtype=torch.float32)  # 0 = same
        else:
            cls_b = np.random.choice([c for c in range(10) if c != cls_a])
            idx_b = np.random.choice(self.class_indices[cls_b])
            y     = torch.tensor(1, dtype=torch.float32)  # 1 = different

        # (1, 28, 28)
        img_a = self.data[idx_a].unsqueeze(0)
        img_b = self.data[idx_b].unsqueeze(0)

        return img_a, img_b, y


pair_train = PairDataset("train", num_pairs=8000)
pair_val   = PairDataset("test",  num_pairs=2000)
pair_loader_tr = DataLoader(pair_train, batch_size=64, shuffle=True)
pair_loader_vl = DataLoader(pair_val,   batch_size=64, shuffle=False)

img_a, img_b, y = pair_train[0]
print(f"img_a : {img_a.shape},  img_b : {img_b.shape},  label y={y.item()} (0=same, 1=diff)")

In [ ]:
class EmbeddingNet(nn.Module):
    """
    Shared encoder for Siamese and Triplet networks.

    Maps a greyscale 28×28 image to a normalised d_embed-dimensional vector.
    L2 normalisation (last line) projects outputs onto the unit hypersphere,
    which makes cosine similarity equivalent to dot product and simplifies
    the distance geometry.
    """

    def __init__(self, embed_dim: int = 64) -> None:
        super().__init__()

        # Convolutional feature extractor
        # (batch_num, 1, 28, 28) → (batch_num, 64, 7, 7)
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )

        # Projection head: flatten → MLP → unit vector
        # (batch_num, 64*7*7) → (batch_num, embed_dim)
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, 1, 28, 28) → (batch_num, 64, 7, 7)
        h = self.features(x)
        # (batch_num, 64*7*7) → (batch_num, embed_dim)
        z = self.proj(h)
        # Project onto unit hypersphere — cosine dist = Euclidean dist on sphere
        # (batch_num, embed_dim) → (batch_num, embed_dim)
        return F.normalize(z, p=2, dim=1)


class ContrastiveLoss(nn.Module):
    """
    Contrastive loss for Siamese network training.

    Args:
        margin : Minimum distance for dissimilar pairs (default 1.0 for unit sphere)
    """

    def __init__(self, margin: float = 1.0) -> None:
        super().__init__()
        self.margin = margin

    def forward(
        self, z_a: torch.Tensor, z_b: torch.Tensor, y: torch.Tensor
    ) -> torch.Tensor:
        """
        Args:
            z_a, z_b : (batch_num, embed_dim) normalised embeddings
            y        : (batch_num,) labels — 0 = same class, 1 = different class

        Returns:
            loss : scalar mean contrastive loss
        """
        # Euclidean distance between paired embeddings
        # (batch_num, embed_dim) → (batch_num,)
        dist = F.pairwise_distance(z_a, z_b, p=2)

        # Same pair: push distance to 0 → (1-y) * d^2 / 2
        same_loss  = (1 - y) * dist.pow(2) / 2
        # Diff pair: push distance ≥ margin → y * max(0, m-d)^2 / 2
        diff_loss  = y * F.relu(self.margin - dist).pow(2) / 2

        return (same_loss + diff_loss).mean()


def train_siamese(model, loader, val_loader, n_epochs=5, lr=1e-3):
    """Trains Siamese network and returns val accuracy."""
    model.to(DEVICE)
    criterion = ContrastiveLoss(margin=1.0)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    history   = {"train_loss": [], "val_acc": []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss = 0.0
        for img_a, img_b, y in loader:
            img_a, img_b, y = img_a.to(DEVICE), img_b.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            # (batch_num, 1, 28, 28) → (batch_num, embed_dim)
            z_a, z_b = model(img_a), model(img_b)
            loss = criterion(z_a, z_b, y)
            loss.backward(); optimizer.step()
            running_loss += loss.item() * img_a.size(0)

        model.eval()
        correct = 0
        threshold = 0.5  # predict "same" if distance < threshold
        with torch.no_grad():
            for img_a, img_b, y in val_loader:
                img_a, img_b, y = img_a.to(DEVICE), img_b.to(DEVICE), y.to(DEVICE)
                z_a, z_b = model(img_a), model(img_b)
                dist     = F.pairwise_distance(z_a, z_b)
                pred     = (dist > threshold).float()
                correct += (pred == y).sum().item()

        tl  = running_loss / len(loader.dataset)
        acc = correct / len(val_loader.dataset)
        history["train_loss"].append(tl)
        history["val_acc"].append(acc)
        print(f"Epoch {epoch}/{n_epochs} | loss={tl:.4f} | val_acc={acc:.3f}")

    return history


torch.manual_seed(0)
siamese_net     = EmbeddingNet(embed_dim=64)
history_siamese = train_siamese(siamese_net, pair_loader_tr, pair_loader_vl, n_epochs=5)

# 2) Triplet Loss

**The problem with contrastive loss**
Contrastive loss treats pairs independently. Easy negatives (clearly different
classes) contribute zero gradient once they are far enough apart, leading to
slow convergence in large-scale settings.

**Triplet loss** (Schroff et al., FaceNet, 2015)

Trains on *triplets* $(a, p, n)$:
- $a$ = **anchor** (reference sample)
- $p$ = **positive** (same class as anchor)
- $n$ = **negative** (different class from anchor)

$$L_{triplet} = \max(0,\; \|f(a)-f(p)\|^2 - \|f(a)-f(n)\|^2 + \alpha)$$

The margin $\alpha$ prevents the trivial solution $f(x) = 0$ for all inputs
by requiring positives to be strictly closer than negatives.

**Hard negative mining**
Random triplets are mostly *easy* (negative already far from anchor).
Hard mining selects:
- **Hard positive**: same class but highest distance from anchor
- **Hard negative**: different class but *smallest* distance from anchor (closest wrong class)

In [ ]:
class TripletDataset(Dataset):
    """
    Generates triplets (anchor, positive, negative) from MNIST.
    Uses batch-level hard mining: for each anchor, find the hardest
    positive and negative within the current mini-batch.
    """

    def __init__(self, split: str = "train", num_triplets: int = 10000) -> None:
        transform = transforms.ToTensor()
        base = torchvision.datasets.MNIST(
            root="./data", train=(split == "train"), download=True, transform=transform
        )
        self.data   = base.data.float() / 255.0
        self.labels = base.targets

        # Index by class for fast positive/negative sampling
        self.class_idx = {
            c: (self.labels == c).nonzero(as_tuple=True)[0].tolist()
            for c in range(10)
        }
        self.num_triplets = num_triplets

    def __len__(self): return self.num_triplets

    def __getitem__(self, _) -> tuple:
        # Sample anchor and its class
        cls_a  = np.random.randint(10)
        idx_a  = np.random.choice(self.class_idx[cls_a])
        # Positive: same class, different index
        idx_p  = np.random.choice(self.class_idx[cls_a])
        # Negative: different class
        cls_n  = np.random.choice([c for c in range(10) if c != cls_a])
        idx_n  = np.random.choice(self.class_idx[cls_n])

        # (1, 28, 28) for each
        return (
            self.data[idx_a].unsqueeze(0),
            self.data[idx_p].unsqueeze(0),
            self.data[idx_n].unsqueeze(0),
        )


class TripletLoss(nn.Module):
    """
    Standard triplet loss with margin.

    Args:
        margin : Minimum separation between positive and negative distances
    """

    def __init__(self, margin: float = 0.3) -> None:
        super().__init__()
        self.margin = margin

    def forward(
        self,
        z_a: torch.Tensor,
        z_p: torch.Tensor,
        z_n: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            z_a : (batch_num, embed_dim) anchor embeddings
            z_p : (batch_num, embed_dim) positive embeddings
            z_n : (batch_num, embed_dim) negative embeddings

        Returns:
            loss : Scalar mean triplet loss
        """
        # Squared Euclidean distances
        # (batch_num, embed_dim) → (batch_num,)
        dist_pos = (z_a - z_p).pow(2).sum(dim=1)
        dist_neg = (z_a - z_n).pow(2).sum(dim=1)

        # Hinge: require d(a,n) > d(a,p) + margin
        # (batch_num,)
        losses = F.relu(dist_pos - dist_neg + self.margin)

        # Fraction of non-zero (active) triplets — diagnostic for hard mining
        active_frac = (losses > 0).float().mean().item()
        # Store as attribute so training loop can log it
        self.active_frac = active_frac

        return losses.mean()


def train_triplet(model, loader, val_loader, n_epochs=5, lr=1e-3):
    model.to(DEVICE)
    criterion = TripletLoss(margin=0.3)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    history   = {"train_loss": [], "active_frac": [], "val_acc": []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss, running_active = 0.0, 0.0
        for z_a, z_p, z_n in loader:
            z_a, z_p, z_n = z_a.to(DEVICE), z_p.to(DEVICE), z_n.to(DEVICE)
            optimizer.zero_grad()
            # (batch_num, 1, 28, 28) → (batch_num, embed_dim)
            ea, ep, en = model(z_a), model(z_p), model(z_n)
            loss = criterion(ea, ep, en)
            loss.backward(); optimizer.step()
            running_loss   += loss.item() * z_a.size(0)
            running_active += criterion.active_frac

        # Val: 1-NN accuracy in embedding space using cosine similarity
        model.eval()
        all_z, all_y = [], []
        with torch.no_grad():
            for imgs, labels in DataLoader(
                torchvision.datasets.MNIST("./data", train=False, download=True,
                    transform=transforms.ToTensor()), batch_size=256):
                all_z.append(model(imgs.to(DEVICE)).cpu())
                all_y.append(labels)
        all_z = torch.cat(all_z)  # (10000, embed_dim)
        all_y = torch.cat(all_y)  # (10000,)

        # Compute pairwise cosine similarity and find nearest neighbour
        sim = all_z @ all_z.T  # (10000, 10000) — fast on unit sphere
        sim.fill_diagonal_(-1)  # exclude self
        nn_idx = sim.argmax(dim=1)
        acc    = (all_y[nn_idx] == all_y).float().mean().item()

        n = len(loader.dataset)
        tl = running_loss / n
        af = running_active / len(loader)
        history["train_loss"].append(tl)
        history["active_frac"].append(af)
        history["val_acc"].append(acc)
        print(f"Epoch {epoch}/{n_epochs} | loss={tl:.4f} | active={af:.2f} | 1NN-acc={acc:.3f}")

    return history


trip_train = TripletDataset("train", 10000)
trip_val   = TripletDataset("test",   2000)
trip_loader_tr = DataLoader(trip_train, batch_size=128, shuffle=True)
trip_loader_vl = DataLoader(trip_val,   batch_size=128, shuffle=False)

torch.manual_seed(0)
triplet_net     = EmbeddingNet(embed_dim=64)
history_triplet = train_triplet(triplet_net, trip_loader_tr, trip_loader_vl, n_epochs=5)

# 3) Embedding Space Visualisation

After training, we project embeddings to 2D with t-SNE to inspect whether
the metric learning objective has produced a well-structured embedding space.

A well-trained metric learner will show distinct, compact clusters per class.

In [ ]:
from sklearn.manifold import TSNE

# Extract embeddings for 1 000 test samples
triplet_net.eval()
sample_imgs, sample_labels_vis = [], []
test_base = torchvision.datasets.MNIST(
    "./data", train=False, download=True, transform=transforms.ToTensor()
)
for img, label in DataLoader(test_base, batch_size=256):
    sample_imgs.append(img)
    sample_labels_vis.append(label)
    if sum(len(x) for x in sample_imgs) >= 1000:
        break

all_imgs_vis = torch.cat(sample_imgs)[:1000]
all_lbls_vis = torch.cat(sample_labels_vis)[:1000]

with torch.no_grad():
    # (1000, 1, 28, 28) → (1000, embed_dim)
    embeddings = triplet_net(all_imgs_vis.to(DEVICE)).cpu().numpy()

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=500)
# (1000, embed_dim) → (1000, 2)
emb_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=all_lbls_vis.numpy(),
                      cmap="tab10", s=10, alpha=0.7)
plt.colorbar(scatter, ticks=range(10), label="MNIST class")
plt.title("t-SNE of triplet-trained embeddings (1 000 test samples)")
plt.axis("off")
plt.tight_layout()
plt.show()

# 4) FaceNet-Style Encoder and facenet-pytorch

**FaceNet** (Schroff et al., 2015) applies triplet loss at scale:
- InceptionResnet-V1 backbone (a deep network with Inception + ResNet modules)
- Hard negative mining across large batches
- L2-normalised 128-dimensional embeddings
- Trained on millions of face images

**ArcFace** (Deng et al., 2019) — the modern standard
ArcFace replaces triplet loss with an additive angular margin softmax loss,
which is more stable and achieves state-of-the-art on face benchmarks.

$$L_{ArcFace} = -\log \frac{e^{s \cos(\theta_{y_i} + m)}}{e^{s\cos(\theta_{y_i}+m)} + \sum_{j \neq y_i} e^{s\cos\theta_j}}$$

This section shows facenet-pytorch (a modern library that provides a pretrained
FaceNet model), without reproducing the full training infrastructure.

In [ ]:
# Install: !pip install facenet-pytorch --quiet

# The facenet_pytorch library provides:
#   1. MTCNN      — multi-task CNN face detector
#   2. InceptionResnetV1 — pretrained FaceNet encoder

# NOTE: uncomment to run after installing facenet-pytorch.

# from facenet_pytorch import MTCNN, InceptionResnetV1
# from PIL import Image
# import requests
# from io import BytesIO

# ── Load detector and encoder ──────────────────────────────────────────────────
# detector = MTCNN(keep_all=True, device=DEVICE)
# encoder  = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)
# # 512-dimensional face embeddings, trained on VGGFace2 (3.3M images, 9 131 identities)
# print(f"Encoder output dim: 512")

# ── Compute face embeddings ────────────────────────────────────────────────────
# def embed_face(image_url: str) -> torch.Tensor:
#     """
#     Detects the largest face in an image and returns its embedding.
#
#     Returns:
#         embedding : (1, 512) L2-normalised face embedding
#     """
#     img    = Image.open(BytesIO(requests.get(image_url, timeout=10).content)).convert("RGB")
#     # MTCNN returns (num_faces, 3, 160, 160) aligned face crops
#     faces  = detector(img)
#     if faces is None:
#         raise ValueError("No face detected")
#     # Use the first detected face
#     # (1, 3, 160, 160) → (1, 512)
#     with torch.no_grad():
#         embedding = encoder(faces[0:1].to(DEVICE))
#     return F.normalize(embedding, p=2, dim=1)

# ── Verify same/different identity ────────────────────────────────────────────
# e1 = embed_face("URL_OF_PERSON_A_IMAGE_1")
# e2 = embed_face("URL_OF_PERSON_A_IMAGE_2")
# e3 = embed_face("URL_OF_PERSON_B_IMAGE")
#
# cosine_same = (e1 * e2).sum().item()   # should be close to 1.0
# cosine_diff = (e1 * e3).sum().item()   # should be significantly lower
# print(f"Same person cosine similarity: {cosine_same:.4f}")
# print(f"Diff person cosine similarity: {cosine_diff:.4f}")

print("facenet-pytorch pseudocode shown.")
print("Install facenet-pytorch and replace URLs with real face images to run.")

# ── Distance threshold for verification ───────────────────────────────────────
print()
print("Standard thresholds for facenet-pytorch VGGFace2 model:")
print("  Cosine similarity > 0.40 → same identity (typical threshold)")
print("  Euclidean distance < 1.0  → same identity (on unit sphere)")
print()
print("The choice of threshold trades off FAR (false accept) vs FRR (false reject).")
print("For security: lower threshold → lower FAR, higher FRR.")

# Summary

| Method | Loss | Metric | Best for |
|---|---|---|---|
| Siamese + Contrastive | $y \cdot d^2 + (1-y) \cdot \max(0, m-d)^2$ | Pairwise distance | Small datasets, binary similarity |
| Triplet | $\max(0, d_+ - d_- + \alpha)$ | Euclidean / cosine | Few-shot learning, face verification |
| ArcFace | Angular margin softmax | Cosine similarity | Large-scale face recognition |

**Key insights for research**

- **Hard mining is essential** at scale: easy negatives contribute zero gradient.
  The `active_frac` logged during training reveals how hard the current batch is.
- **Unit-sphere normalisation** is standard: it decouples magnitude from direction
  and makes distance thresholds interpretable (cosine sim on unit sphere = dot product).
- **The same EmbeddingNet** is used for Siamese and Triplet training —
  only the loss and data pipeline change. This modularity matters for ablation studies.
- **1-NN accuracy** is the cleanest evaluation for embedding quality: it requires
  no threshold tuning and directly measures clustering quality.